In [40]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from datasets import load_dataset
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict

import prompts

In [18]:
## prepare data

# read evidence data
evidence_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_train.csv')
evidence_ls = evidence_df.loc[:,'text'].dropna().to_list()

#read qq data
qq_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv')
qq_ds = Dataset.from_pandas(qq_df.loc[:,['question','follow_up_questions']])

In [21]:
# load embedding model
model_path = '/raid/deallab/SF_RAG_Data/ASQA/models/fine_tuned_model_64'
model = SentenceTransformer(model_path)

# load model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# calculate embeddings for all data
evidence_embeddings = model.encode(evidence_ls, convert_to_tensor=True).to(device)

In [22]:
# load generative model
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

tokenizer_gen.pad_token = tokenizer_gen.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gen.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [83]:
importlib.reload(prompts)

<module 'prompts' from '/home/dataconv/deallab/lasse/sf_rag/sf_rag/prompts.py'>

In [71]:

# retrive docs from the document embeddings
def retrieve_documents(query):
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)

    query_embedding = query_embedding.unsqueeze(0)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10]
    print(top_results)
    res={}
    for idx in top_results:
        tmp=evidence_ls[idx]
        res[tmp]=similarities[idx]
        
    return list(res.keys())

In [78]:
def evaluate_docs(query, docs):
    print(f"Query : {query}")
    print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
        outputs = model_gen.generate(inputs, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=128)
        generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
        
        filter=(generated_text.split('\n')[0])
        print(filter)
        if '#relevant' in filter:
            outs.append((generated_text.split('\n')[1]).strip())
    
    return outs

In [90]:
for entry in qq_ds:
    query = entry['question']
    fu_questions = entry['follow_up_questions']

In [84]:
def make_new_query(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':{input}},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    outputs = model_gen.generate(inputs, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=258)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

In [85]:
query = qq_ds[1]['question']
docs = retrieve_documents(query)
rel_docs = evaluate_docs(query, docs)
make_new_query(query, rel_docs)

tensor([340, 356, 384, 377, 350, 360, 361, 362, 363, 374], device='cuda:0')
Query : Who won the ncaa football national championship played in 2016?
----------------------------------------------------------------------------------------------------
#irrelevant
#irrelevant
#irrelevant
#relevant
#irrelevant
#relevant
#irrelevant


"['Who played against Washington in the 2016 NCAA Football National Championship?',\n    'Who was the runner-up in the 2016 NCAA Football National Championship?',\n    'What was the score of the 2016 NCAA Football National Championship between Alabama and Washington?',\n    'Which team, Alabama or Washington, had the higher score in the 2016 NCAA Football National Championship?',\n    'Who was the actual winner of the 2016 NCAA Football National Championship, contrary to the information provided?',\n    'What was the final score of the 2016 NCAA Football National Championship game between Clemson and Alabama?']"